# DINOv2-S/14-Reg → OpenVINO (FP32 / FP16 / INT8): step-by-step, end-to-end

A **standalone** notebook that converts the **DINOv2-S/14-Reg** backbone
([`facebook/dinov2-with-registers-small`](https://huggingface.co/facebook/dinov2-with-registers-small))
to OpenVINO and compares every precision against the original PyTorch model on a single sample
image — throughout, on **dense per-patch features**, the representation DINOv2 uses for dense
tasks (segmentation, correspondence, PCA).

**What this notebook does, in order:**
1. Install every dependency into the active Jupyter kernel (a dedicated virtual environment is recommended); packages already present are detected and skipped.
2. Download the DINOv2-S/14-Reg weights **once** into `checkpoints/` and load them.
3. Take **one sample image** for the demo — a local COCO `val2017` image when that
   folder is present, otherwise the well-known COCO dog image — cached **once** in `data/`.
4. Convert the model to OpenVINO IR in **FP32**, **FP16**, **INT8-PTQ** and **INT8-WOQ**
   under `ov_models/`: FP32 comes from the reference graph; **FP16 and both INT8 graphs are
   built from an optimized export graph** (split Q/K/V, fused token prefix, direct <spell>SDPA</spell>)
   traced directly in FP16. **INT8-PTQ** is NNCF post-training quantization — a strided
   subset of calibration images (ImageNet-val class folders or a COCO `val2017` folder,
   whichever is present; nothing is downloaded for it) calibrates the quantizers with the
   repository's shipped recipe (MIXED preset, GPU target, transformer-aware). **INT8-WOQ**
   is weights-only INT8 (`nncf.compress_weights`), which needs no calibration data and is
   always produced.
5. Run **each** OpenVINO model end-to-end on the sample image.
6. Quantify fidelity with the measure the DINO lineage uses to characterize dense features —
   **cosine similarity**, complemented by **mean squared error** and **MAE**.
7. Report **latency** for each engine.
8. Visualize the dense features with **PCA**, **segmentation** and a per-patch
   **cosine-similarity** map (torch vs each precision).

> **Everything is cached.** Weights, the sample image and the exported IRs are written once and
> reused: a re-run skips the downloads and the conversion — and prints that it did — so only
> inference, metrics and plots are recomputed.
> **INT8 calibration set.** PTQ draws a strided subset from the first calibration folder
> that exists: `datasets/imagenet/val` (ImageNet-val class folders) or `datasets/coco/val2017`
> (flat COCO images); point `CALIB_DIR`/`CALIB_KIND` at your own copy in the configuration
> cell. No calibration images are ever downloaded. If neither folder is present, INT8-PTQ is
> skipped and **INT8-WOQ** (weights-only, no data needed) still runs, so the notebook works
> end-to-end out of the box.

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

#### Table of contents:

- [1. Install dependencies](#1.-Install-dependencies)
- [2. Imports & configuration](#2.-Imports-&-configuration)
- [3. Download & load the DINOv2-S/14-Reg model](#3.-Download-&-load-the-DINOv2-S/14-Reg-model)
- [4. Get a sample image (COCO)](#4.-Get-a-sample-image-(COCO))
- [5. Torch reference features (ground truth)](#5.-Torch-reference-features-(ground-truth))
- [6. OpenVINO conversion → FP32 / FP16 / INT8 (NNCF PTQ + WOQ)](#6.-OpenVINO-conversion-→-FP32-/-FP16-/-INT8-(NNCF-PTQ-+-WOQ))
- [7. Run each OpenVINO model on the sample image](#7.-Run-each-OpenVINO-model-on-the-sample-image)
- [8. Fidelity metrics vs the torch model](#8.-Fidelity-metrics-vs-the-torch-model)
- [9. Latency](#9.-Latency)
- [10. Visualization — PCA & segmentation (torch vs OpenVINO)](#10.-Visualization-—-PCA-&-segmentation-(torch-vs-OpenVINO))
- [11. Summary](#11.-Summary)

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/dinov2-s-reg/dinov2-s-reg.ipynb" />

## 1. Install dependencies

Create the virtual environment once (from this notebook's folder) and register it as a Jupyter
kernel:

```bash
python3 -m venv .dinov2sreg-nb-venv
./.dinov2sreg-nb-venv/bin/python -m pip install ipykernel
./.dinov2sreg-nb-venv/bin/python -m ipykernel install --user --name dinov2sreg-nb-venv \
    --display-name "Python (.dinov2sreg-nb-venv)"
```

Then select the `Python (.dinov2sreg-nb-venv)` kernel. The cell below is **self-contained**:
it pip-installs everything the notebook needs directly into the running kernel, following the
OpenVINO-notebook methodology (a small `pip_install` helper that shells out to the active
interpreter).

- **torch / torchvision** are installed from the **CPU index**
  (`--extra-index-url https://download.pytorch.org/whl/cpu`) — the PyTorch reference runs on
  CPU; the OpenVINO engines use the Intel GPU/CPU directly.
- Re-runs are fast: pip skips packages that are already satisfied.

> **GPU note.** The OpenVINO device is auto-selected in the next section. If the Intel GPU's
> Level Zero driver aborts device enumeration (a known driver/kernel mismatch on some
> systems), the notebook detects it and falls back to **CPU** automatically — the rest of the
> notebook runs unchanged.

In [ ]:
# === Install dependencies into the active Jupyter kernel =====================
# This notebook is self-contained: it installs what it needs into the running kernel.
# Every package is probed with ``importlib`` first and only the missing ones are
# pip-installed, so re-runs are fast and an environment that already has the stack
# (for example the openvino_notebooks ``openvino_env``) is left untouched.
import glob
import importlib
import importlib.metadata
import importlib.util
import subprocess
import sys


def pip_install(*args):
    """Install packages into the active kernel (OpenVINO-notebook methodology)."""
    cli = []
    for a in args:
        cli.extend(str(a).split(" "))
    subprocess.run([sys.executable, "-m", "pip", "install", *cli], check=True)


# Recommend a virtual environment (openvino_notebooks practice); warn, never fail,
# because CI runs this notebook inside a managed environment on purpose.
if sys.prefix == sys.base_prefix:
    print("note: the kernel is not inside a virtual environment; installing into it anyway")

# --- torch / torchvision -------------------------------------------------------
# The latency benchmark times the PyTorch reference on CPU always, and additionally
# on the Intel GPU through torch.xpu when the build provides it. On a machine with
# an Intel GPU (i915/xe kernel driver bound) we therefore make sure an XPU build is
# in place: a missing torch is installed from the XPU wheel index, and a CPU-only
# build is replaced by the matching XPU build. Without an Intel GPU the CPU build
# is used; the OpenVINO engines never need torch.
# Detection stays metadata-only (no ``import torch``), so a swapped-in build is the
# first one this kernel process ever imports.
_intel_gpu = bool(glob.glob("/sys/bus/pci/drivers/xe/0000:*") + glob.glob("/sys/bus/pci/drivers/i915/0000:*"))


def _torch_version():
    try:
        return importlib.metadata.version("torch")
    except importlib.metadata.PackageNotFoundError:
        return None


_have_torch = bool(importlib.util.find_spec("torch") and importlib.util.find_spec("torchvision"))
if _have_torch and _intel_gpu and "+xpu" not in (_torch_version() or ""):
    print(f"torch {_torch_version()} has no XPU support; replacing it with the Intel GPU (XPU) build")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torch", "torchvision"], check=True)
    _have_torch = False
if not _have_torch:
    if _intel_gpu:
        pip_install("-q", "--index-url", "https://download.pytorch.org/whl/xpu", "torch", "torchvision")
    else:
        pip_install("-q", "--extra-index-url", "https://download.pytorch.org/whl/cpu", "torch", "torchvision")
if "torch" in sys.modules and _intel_gpu and "+xpu" not in (_torch_version() or ""):
    print("note: restart the kernel to load the newly installed torch build")

# --- the rest of the runtime: import name -> pip spec --------------------------
# Only packages whose import probe fails are installed, in a single pip call.
REQUIRED = {
    "openvino": "openvino>=2025.1",
    "nncf": "nncf>=2.13",
    "transformers": "transformers>=4.56",
    "huggingface_hub": "huggingface_hub>=0.24",
    "PIL": "pillow",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
    "requests": "requests",
    "ipywidgets": "ipywidgets",
}
missing = [spec for name, spec in REQUIRED.items() if importlib.util.find_spec(name) is None]
if missing:
    pip_install("-q", *missing)
else:
    print("all runtime packages already present")

import importlib

print("\ndependencies ready in", sys.executable)
for m in ["openvino", "nncf", "torch", "torchvision", "transformers", "huggingface_hub", "PIL", "numpy", "matplotlib", "sklearn", "requests", "ipywidgets"]:
    try:
        mod = importlib.import_module(m)
        print(f"  {m:<16} {getattr(mod, '__version__', 'ok')}")
    except Exception as exc:
        print(f"  {m:<16} MISSING -> {exc}")

# --- telemetry (openvino_notebooks CI) ----------------------------------
# Fetch the shared helper and report that this notebook was run.
import urllib.request
from pathlib import Path

if not Path("notebook_utils.py").exists():
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
        "notebook_utils.py",
    )

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("dinov2-s-reg.ipynb")

## 2. Imports & configuration

Central knobs: the Hugging Face model id, the evaluation resolution, and the OpenVINO device
(auto-selects the Intel **GPU** if present, else **CPU**).

Three folders are created next to the notebook and **reused across runs** — this is what makes
the notebook cheap to re-run:

| folder | contents | re-run behaviour |
|---|---|---|
| `checkpoints/` | DINOv2-S/14-Reg torch weights | download skipped if present |
| `data/` | the sample image | download skipped if present |
| `ov_models/` | `fp32/`, `fp16/`, `int8/` OpenVINO IRs | conversion skipped if present |

In [ ]:
import json
import shutil
import subprocess
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import nncf
import numpy as np
import openvino as ov
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
from transformers import AutoImageProcessor, AutoModel

torch.manual_seed(0)
torch.set_grad_enabled(False)  # inference only

# --- model source -------------------------------------------------------------
HF_MODEL_ID = "facebook/dinov2-with-registers-small"
HF_TOKEN = None  # set a token string if the HF repo requires authentication

# --- persistent artifact folders (created once, reused on every re-run) -------
REPO_ROOT = Path.cwd()
CHECKPOINT_DIR = REPO_ROOT / "checkpoints" / HF_MODEL_ID.split("/")[-1]
DATA_DIR = REPO_ROOT / "data"
OV_DIR = REPO_ROOT / "ov_models"
for _dir in (CHECKPOINT_DIR, DATA_DIR, OV_DIR):
    _dir.mkdir(parents=True, exist_ok=True)

# --- inference config ---------------------------------------------------------
IMAGE_SIZE = 518  # DINOv2 native resolution (37 x 14-px patches)
BENCH_REPEATS = 20  # timed iterations for latency
PRECISIONS = ("fp32", "fp16", "int8-ptq", "int8-woq")

# --- INT8 calibration ----------------------------------------------------------
# PTQ needs a small set of representative inputs to collect activation statistics.
# The first directory that exists wins: ImageNet-val class folders (the set the
# shipped reference recipe was calibrated on) or a COCO val2017 image folder.
# Nothing is ever downloaded for calibration; with no calibration set present the
# notebook still produces weights-only INT8 (WOQ), which needs no data. Point
# CALIB_DIR / CALIB_KIND at your own copy in this cell to override. A strided
# subset spans the whole set, and every image goes through the same transform
# used at inference time, so calibration ranges match evaluation.
# The shared datasets live at the repository root, so they are looked up there
# too (the notebook runs from its own folder, one or two levels below the root).
_imagenet_candidates = [
    REPO_ROOT / "datasets" / "imagenet" / "val",
    REPO_ROOT.parent / "datasets" / "imagenet" / "val",
    REPO_ROOT.parent.parent / "datasets" / "imagenet" / "val",
]
_coco_candidates = [
    REPO_ROOT / "datasets" / "coco" / "val2017",
    REPO_ROOT.parent / "datasets" / "coco" / "val2017",
    REPO_ROOT.parent.parent / "datasets" / "coco" / "val2017",
]
if any(p.exists() for p in _imagenet_candidates):
    CALIB_DIR, CALIB_KIND = next(p for p in _imagenet_candidates if p.exists()), "imagenet"
elif any(p.exists() for p in _coco_candidates):
    CALIB_DIR, CALIB_KIND = next(p for p in _coco_candidates if p.exists()), "coco"
else:
    CALIB_DIR, CALIB_KIND = None, None
CALIB_SAMPLES = 512  # images used for PTQ calibration (strided subset)
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def probe_ov_devices():
    """Return OpenVINO's available devices, or None if the probe crashes.

    On some systems the Level Zero (GPU) driver aborts the *process* (SIGABRT)
    when OpenVINO enumerates devices. A SIGABRT cannot be caught in Python, so
    the probe runs in a short-lived subprocess: if it dies, we fall back to CPU
    instead of taking the whole kernel down with it.
    """
    code = "import openvino as ov, json, sys;" "print(json.dumps(ov.Core().available_devices))"
    try:
        r = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True, timeout=120)
    except Exception:
        return None
    if r.returncode != 0:
        return None
    for line in reversed(r.stdout.splitlines()):
        line = line.strip()
        if line.startswith("["):
            try:
                return json.loads(line)
            except Exception:
                return None
    return None


_devices = probe_ov_devices()
if _devices and "GPU" in _devices:
    DEVICE = "GPU"
    print(f"OpenVINO devices: {_devices}  ->  using '{DEVICE}'")
else:
    DEVICE = "CPU"
    if _devices is None:
        print("OpenVINO device probe aborted (GPU driver issue) -> using 'CPU'")
    else:
        print(f"OpenVINO devices: {_devices}  ->  no GPU, using 'CPU'")

print(f"Model id     : {HF_MODEL_ID}")
print(f"Image size   : {IMAGE_SIZE}px")
print(f"checkpoints  : {CHECKPOINT_DIR}")
print(f"sample data  : {DATA_DIR}")
print(f"OpenVINO IRs : {OV_DIR}")
if CALIB_DIR is None:
    print("INT8 calib   : none found -> INT8-PTQ skipped, INT8-WOQ (weights-only) still produced")
else:
    print(f"INT8 calib   : {CALIB_DIR} ({CALIB_SAMPLES} images, kind={CALIB_KIND})")

In [ ]:
import ipywidgets as widgets

# Device selection widget. The options come from the subprocess probe in the
# cell above (which guards against the Level Zero SIGABRT), so we never call
# core.available_devices directly in this kernel.
_device_options = list(_devices) if _devices else ["CPU"]
if "AUTO" not in _device_options:
    _device_options = ["AUTO"] + _device_options

device = widgets.Dropdown(
    options=_device_options,
    value=DEVICE,  # default to the probed choice from the cell above
    description="Device:",
    layout=widgets.Layout(width="320px"),
)
display(device)

# Resolve the selection: AUTO -> GPU when available, else CPU.
DEVICE = "GPU" if (device.value == "AUTO" and "GPU" in _device_options) else device.value
print(f"Selected OpenVINO device: {DEVICE}")

## 3. Download & load the DINOv2-S/14-Reg model

The weights are pulled from the Hugging Face Hub **into `checkpoints/`** on the first run and
loaded from there afterwards — the download is skipped when the checkpoint is already on disk,
and falls back to the local Hub cache if the network is unavailable.
[`facebook/dinov2-with-registers-small`](https://huggingface.co/facebook/dinov2-with-registers-small)
is the Hugging Face port of the original
[dinov2_vits14_reg4](https://github.com/facebookresearch/dinov2) checkpoint (verified
bit-identical weights), so no gated access is required.

The backbone is then wrapped twice:

- **`PatchBackbone`** — the *reference* wrapper. DINOv2's token sequence is
  `[CLS] + [4 register tokens] + [patch tokens]`; the wrapper drops the
  `1 + num_register_tokens` prefix and reshapes the patch tokens back onto the spatial grid,
  emitting **dense per-patch features** `(B, D, h, w)`. These dense features are what
  segmentation / PCA consume, and they are the ground truth for every precision below.
- **`OptimizedPatchBackbone`** — an *export-only* rewrite used for the FP16 and INT8 IRs:
  the fused QKV projection becomes three <spell>SDPA</spell>-ready projections, the CLS(+pos) and register
  tokens become one constant prefix concatenated once, and attention calls
  `F.scaled_dot_product_attention` directly. The rewrite is mathematically exact (same
  weights, same graph up to removed concatenations); it is traced in **FP16** so the FP16 IR
  carries true half-precision constants and both INT8 IRs are quantized from the same graph.
  The module paths (`blocks.N.attn.proj`, `patch_embed.proj`, …) are part of the export
  contract: the INT8 recipes keep the matrix multiplications in FP16 by matching those names.

In [ ]:
from huggingface_hub import snapshot_download

CKPT_PATTERNS = ["*.json", "*.txt", "*.safetensors"]


def fetch_checkpoint(repo_id, dst, token=None):
    """Materialize the weights in `dst` once; later runs load them straight from there."""
    # config.json alone is not enough: a partial download can have it without the
    # weights, so require at least one safetensors file before skipping the fetch.
    if (dst / "config.json").exists() and any(dst.glob("*.safetensors")):
        print(f"checkpoint already present -> skipping download  ({dst})")
        return
    print(f"fetching {repo_id}  ->  {dst}")
    try:
        src = snapshot_download(repo_id=repo_id, token=token, allow_patterns=CKPT_PATTERNS)
    except Exception as exc:
        print(f"  Hub unreachable ({type(exc).__name__}) -> falling back to the local Hub cache")
        try:
            src = snapshot_download(repo_id=repo_id, allow_patterns=CKPT_PATTERNS, local_files_only=True)
        except Exception:
            raise RuntimeError(f"Could not obtain '{repo_id}' from the Hub or the local cache.") from exc
    dst.mkdir(parents=True, exist_ok=True)
    for f in Path(src).iterdir():
        if f.is_file():
            shutil.copy(f, dst / f.name)  # follows the cache symlinks -> real files in dst
    print(f"  copied {len(list(dst.glob('*')))} files into {dst}")


fetch_checkpoint(HF_MODEL_ID, CHECKPOINT_DIR, HF_TOKEN)

processor = AutoImageProcessor.from_pretrained(CHECKPOINT_DIR)
backbone = AutoModel.from_pretrained(CHECKPOINT_DIR, attn_implementation="sdpa").eval()

cfg = backbone.config
PATCH = cfg.patch_size
EMBED = cfg.hidden_size
NUM_PREFIX = 1 + cfg.num_register_tokens  # [CLS] + register tokens
assert IMAGE_SIZE % PATCH == 0, f"IMAGE_SIZE must be a multiple of the patch size ({PATCH})"
GRID = IMAGE_SIZE // PATCH
print(f"\npatch_size={PATCH}  hidden={EMBED}  prefix_tokens={NUM_PREFIX}  " f"layers={cfg.num_hidden_layers}")
print(f"dense grid @ {IMAGE_SIZE}px: {GRID}x{GRID} = {GRID * GRID} patches x {EMBED}-d")


class PatchBackbone(nn.Module):
    """DINOv2 backbone -> dense per-patch features (B, D, h, w).

    The token sequence is [CLS] + [register tokens] + [patch tokens]; we drop the
    1 + num_register_tokens prefix and reshape the patch tokens onto the (h, w) grid.
    These dense features are what segmentation / PCA / correspondence consume.
    """

    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

    def forward(self, pixel_values):
        b, _, H, W = pixel_values.shape
        h, w = H // PATCH, W // PATCH
        hid = self.backbone(pixel_values=pixel_values).last_hidden_state
        patch = hid[:, NUM_PREFIX:, :].permute(0, 2, 1)  # (B, D, h*w)
        return patch.reshape(b, EMBED, h, w)  # (B, D, h, w)


patch_model = PatchBackbone(backbone).eval()


class _PatchEmbed(nn.Module):
    """Patch projection under the export-graph module path (``patch_embed.proj``)."""

    def __init__(self, patch_embeddings):
        super().__init__()
        self.proj = patch_embeddings.projection

    def forward(self, pixel_values):
        return self.proj(pixel_values).flatten(2).transpose(1, 2)


class _LayerScale(nn.Module):
    """LayerScale as a plain parameter multiply (identical traced graph)."""

    def __init__(self, gamma):
        super().__init__()
        self.gamma = nn.Parameter(gamma.detach().clone())

    def forward(self, x):
        return x * self.gamma


class _MLP(nn.Module):
    def __init__(self, mlp):
        super().__init__()
        self.fc1, self.act, self.fc2 = mlp.fc1, mlp.activation_fn, mlp.fc2

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))


class _SplitQKVAttention(nn.Module):
    """Attention calling SDPA directly through the three projection Linears.

    The projections are the original modules, so the weights are untouched; the
    output projection is registered as ``attn.proj`` so the INT8 ignored scope
    (``attn.proj`` stays in floating point) matches the graph by name.
    """

    def __init__(self, attention):
        super().__init__()
        self.num_heads = attention.num_attention_heads
        self.q, self.k, self.v = attention.q_proj, attention.k_proj, attention.v_proj
        self.proj = attention.o_proj

    def forward(self, tokens):
        batch, length, channels = tokens.shape
        heads, head_dim = self.num_heads, channels // self.num_heads
        q = self.q(tokens).reshape(batch, length, heads, head_dim).transpose(1, 2)
        k = self.k(tokens).reshape(batch, length, heads, head_dim).transpose(1, 2)
        v = self.v(tokens).reshape(batch, length, heads, head_dim).transpose(1, 2)
        out = F.scaled_dot_product_attention(q, k, v)
        return self.proj(out.transpose(1, 2).reshape(batch, length, channels))


class _OptimizedBlock(nn.Module):
    """One encoder layer: direct SDPA plus LayerScale as plain multiplies."""

    def __init__(self, layer):
        super().__init__()
        self.norm1, self.norm2 = layer.norm1, layer.norm2
        self.attn = _SplitQKVAttention(layer.attention)
        self.mlp = _MLP(layer.mlp)
        self.ls1 = _LayerScale(layer.layer_scale1.lambda1)
        self.ls2 = _LayerScale(layer.layer_scale2.lambda1)

    def forward(self, tokens):
        tokens = tokens + self.ls1(self.attn(self.norm1(tokens)))
        return tokens + self.ls2(self.mlp(self.norm2(tokens)))


class _ShimAttention(nn.Module):
    """Name-only view that registers the output projection as ``attn.proj``."""

    def __init__(self, o_proj):
        super().__init__()
        self.proj = o_proj


class _ShimBlock(nn.Module):
    def __init__(self, layer):
        super().__init__()
        self.attn = _ShimAttention(layer.attention.o_proj)


class _FrozenBackbone(nn.Module):
    """Token-side parameters and the final norm, under the export-graph names.

    The shim ``blocks`` list exists only so the traced constants are named
    ``self.backbone.blocks.N.attn.proj.weight``: the INT8 ignored scope matches
    GEMMs by those names, so the module path is part of the export contract.
    """

    def __init__(self, hf):
        super().__init__()
        self.blocks = nn.ModuleList(_ShimBlock(layer) for layer in hf.encoder.layer)
        self.patch_embed = _PatchEmbed(hf.embeddings.patch_embeddings)
        self.cls_token = nn.Parameter(hf.embeddings.cls_token.detach().clone())
        self.register_tokens = nn.Parameter(hf.embeddings.register_tokens.detach().clone())
        self.pos_embed = nn.Parameter(hf.embeddings.position_embeddings.detach().clone())
        self.norm = hf.layernorm

    def interpolate_pos_encoding(self, x, width, height):
        previous_dtype = x.dtype
        npatch = x.shape[1] - 1
        n_stored = self.pos_embed.shape[1] - 1
        if npatch == n_stored and width == height:
            return self.pos_embed  # native resolution: no interpolation (the 518-px path)
        pos_embed = self.pos_embed.float()
        class_pos_embed = pos_embed[:, 0]
        patch_pos_embed = pos_embed[:, 1:]
        dim = x.shape[-1]
        w0, h0 = width // PATCH, height // PATCH
        side = int(n_stored**0.5)
        # Historical kludge (facebookresearch/dino#8): a small offset keeps the
        # bicubic scale factor off exact bin boundaries.
        sx = float(w0 + 0.1) / side
        sy = float(h0 + 0.1) / side
        patch_pos_embed = F.interpolate(
            patch_pos_embed.reshape(1, side, side, dim).permute(0, 3, 1, 2),
            mode="bicubic",
            antialias=False,
            scale_factor=(sx, sy),
        )
        patch_pos_embed = patch_pos_embed.permute(0, 2, 3, 1).reshape(1, -1, dim)
        return torch.cat((class_pos_embed.unsqueeze(0), patch_pos_embed), dim=1).to(previous_dtype)


class OptimizedPatchBackbone(nn.Module):
    """Export-only rewrite of the DINOv2-S/14-Reg backbone -> (B, D, h, w).

    * Split Q/K/V projections writing SDPA-ready tensors (exact; the weights are
      the original modules).
    * Fused token prefix: CLS(+pos) and the register tokens are one constant
      prefix concatenated once (removes a concat and a separate CLS add).
    * Direct F.scaled_dot_product_attention (the SDPA the GPU plugin fuses).
    * Same weights as the reference; the rewrite is exact.
    """

    def __init__(self, backbone):
        super().__init__()
        self.backbone = _FrozenBackbone(backbone)  # registered first -> constant names
        self.blocks = nn.ModuleList(_OptimizedBlock(layer) for layer in backbone.encoder.layer)
        self.num_prefix = 1 + cfg.num_register_tokens
        self.patch_size = PATCH
        self.embed_dim = EMBED

    def forward(self, pixel_values):
        batch, _, height, width = pixel_values.shape
        patches = self.backbone.patch_embed(pixel_values)
        probe = patches.new_empty((1, patches.shape[1] + 1, patches.shape[2]))
        pos = self.backbone.interpolate_pos_encoding(probe, width, height).to(patches.dtype)
        prefix = self.backbone.cls_token + pos[:, :1]
        prefix = torch.cat((prefix, self.backbone.register_tokens), dim=1)
        tokens = torch.cat((prefix.expand(batch, -1, -1), patches + pos[:, 1:]), dim=1)
        for block in self.blocks:
            tokens = block(tokens)
        tokens = self.backbone.norm(tokens)
        patches = tokens[:, self.num_prefix :]
        grid = (height // self.patch_size, width // self.patch_size)
        return patches.reshape(batch, *grid, self.embed_dim).permute(0, 3, 1, 2)


import copy

opt_model = OptimizedPatchBackbone(copy.deepcopy(backbone).to(torch.float16)).eval()


def build_transform(processor, image_size):
    """Paper-faithful val transform: resize short side, center-crop, normalize."""
    return T.Compose(
        [
            T.Resize(image_size, interpolation=T.InterpolationMode.BICUBIC),
            T.CenterCrop(image_size),
            T.ToTensor(),
            T.Normalize(mean=processor.image_mean, std=processor.image_std),
        ]
    )


transform = build_transform(processor, IMAGE_SIZE)
print("reference + optimized wrappers ready")

## 4. Get a sample image (COCO)

We evaluate on the same natural image the DINO examples use — the COCO *dog* photo — so the
results are directly comparable across the DINOv2 / DINOv3 notebooks. It is downloaded **once**
into `data/` and reused on every later run, so the notebook stays self-contained without
re-fetching anything.

In [ ]:
# One demo image, cached in data/. Preference order: an already-cached copy, a
# deterministic local COCO val2017 image (no download), and only then the stable
# well-known COCO dog image used across the PyTorch / DINO examples.
SAMPLE_URL = "https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg"
SAMPLE_PATH = DATA_DIR / "sample.jpg"

if SAMPLE_PATH.exists():
    print(f"sample image already present -> reusing it  ({SAMPLE_PATH})")
else:
    _coco_dir = next((p for p in _coco_candidates if p.exists()), None)
    _coco_images = sorted(p for p in _coco_dir.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS) if _coco_dir else []
    if _coco_images:
        shutil.copy(_coco_images[0], SAMPLE_PATH)
        print(f"using local COCO image {_coco_images[0].name}  ->  {SAMPLE_PATH}")
    else:
        resp = requests.get(SAMPLE_URL, timeout=30)
        resp.raise_for_status()
        SAMPLE_PATH.write_bytes(resp.content)
        print(f"downloaded {SAMPLE_URL}  ->  {SAMPLE_PATH}")

pil_image = Image.open(SAMPLE_PATH).convert("RGB")

# Model input + a display image (exactly the pixels the model sees).
pixel_values = transform(pil_image).unsqueeze(0)  # (1, 3, H, W)
_disp = T.CenterCrop(IMAGE_SIZE)(T.Resize(IMAGE_SIZE, interpolation=T.InterpolationMode.BICUBIC)(pil_image))
display_np = np.asarray(_disp)
print("input tensor:", tuple(pixel_values.shape))

plt.figure(figsize=(3.6, 3.6))
plt.imshow(display_np)
plt.title("sample input")
plt.axis("off")
plt.show()

## 5. Torch reference features (ground truth)

Run the PyTorch model once. Its dense features are the **reference** every OpenVINO precision
is compared against (both visually and via metrics).

In [ ]:
with torch.inference_mode():
    ref_torch = patch_model(pixel_values)  # (1, D, h, w)
print("torch dense features:", tuple(ref_torch.shape))


def to_rows(feat_bdhw):
    """(1, D, h, w) -> (h*w, D) rows, one row per patch, aligned across engines."""
    return feat_bdhw[0].movedim(0, -1).reshape(-1, EMBED).float()


ref_rows = to_rows(ref_torch)  # (h*w, D)
ref_hwd = ref_torch[0].movedim(0, -1).float()  # (h, w, D) for PCA/seg

## 6. OpenVINO conversion → FP32 / FP16 / INT8 (NNCF PTQ + WOQ)

The IRs live in `ov_models/<precision>/` and are named after the input resolution and the
**patch** output variant (dense per-patch features, not the CLS-only `base` variant). **The
cell first checks what is already on disk and converts only what is missing**, printing either
`skipping conversion` or `converted` for each precision — so a second run of the notebook goes
straight to inference, while changing `IMAGE_SIZE` transparently triggers a fresh conversion.

Three precisions are derived from two source graphs traced from the torch wrappers:

- **FP32** — the *reference* graph traced in FP32; full-precision constants. This is the
  unmodified control row.
- **FP16** — the *optimized* graph (fused token prefix, direct <spell>SDPA</spell>) traced **directly in
  FP16**, so the constants are true half precision rather than a downcast of an FP32 graph.
- **INT8-PTQ** — **NNCF post-training quantization** of that same optimized FP16 graph, with
  the recipe the repository ships (`ov_enab/ov_quantize.py`): **MIXED** preset (symmetric
  weights, asymmetric activations), `target_device=GPU`, `model_type=TRANSFORMER` (<spell>SDPA</spell> stays
  FP16 and <spell>SmoothQuant</spell> is enabled — quantizing attention collapses ViT features), fast bias
  correction, and the `hybrid-b0fc1` scope: the patch-embedding convolution, the attention
  output projections, the MLP output projections and the first block's MLP input projection
  stay in FP16 (the residual-feeding matrix multiplications). A strided subset of `CALIB_SAMPLES` calibration
  images (ImageNet-val class folders or COCO `val2017`, same transform as inference) provides
  the activation statistics. The <spell>SmoothQuant</spell> scales that feed only FP16 matrix multiplications are then
  folded into the weights, removing standalone element-wise kernels. This IR is skipped when
  no calibration folder is present (calibration images are never downloaded).
- **INT8-WOQ** — **weights-only quantization** (`nncf.compress_weights`, INT8 symmetric
  weights, dynamic activation quantization in the plugin, attention output projections and
  block-0 `fc1` kept in FP16). It needs no calibration data, so it is always produced — and it
  is the fallback INT8 engine when PTQ has no images to calibrate on.

In [ ]:
from torch.utils.data import Dataset

# The IR is exported for a fixed input shape, so the resolution is part of its name:
# changing IMAGE_SIZE transparently triggers a fresh conversion instead of reusing a
# stale graph. "patch" = dense per-patch features (the base/CLS-only variant is not used).
IR_STEM = f"dinov2sreg_vits14_patch_{IMAGE_SIZE}"
ir_paths = {p: OV_DIR / p / f"{IR_STEM}.xml" for p in PRECISIONS}


def ir_on_disk(path):
    """An IR is usable only if both the graph (.xml) and the weights (.bin) exist."""
    return path.exists() and path.with_suffix(".bin").exists()


class _ImageListDataset(Dataset):
    """Fixed list of image files (flat calibration folders such as COCO val2017)."""

    def __init__(self, paths, transform):
        self.paths, self.transform = list(paths), transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        with Image.open(self.paths[idx]) as img:
            return self.transform(img.convert("RGB"))


def build_calibration(image_dir, kind, transform, num_samples, offset=1):
    """Strided subset of calibration images -> nncf.Dataset of raw numpy inputs.

    ``kind="imagenet"`` walks the class folders (sorted classes, sorted files);
    ``kind="coco"`` walks a flat image folder. The stride spans the whole set;
    ``offset`` shifts it so the calibration images differ from the evaluation
    subsets (which start at index 0). Every image goes through the same
    transform used at inference time.
    """
    from torch.utils.data import DataLoader, Subset

    if kind == "imagenet":
        from torchvision.datasets import ImageFolder

        full = ImageFolder(image_dir, transform=transform)
    else:
        files = sorted(p for p in Path(image_dir).iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS)
        full = _ImageListDataset(files, transform)

    stride = max(1, len(full) // max(1, num_samples))
    indices = list(range(offset % stride, len(full), stride))[:num_samples]
    ds = Subset(full, indices)
    loader = DataLoader(ds, batch_size=1, shuffle=False)

    def transform_fn(batch):
        images = batch[0] if isinstance(batch, (list, tuple)) else batch
        return images.numpy()

    if kind == "imagenet":
        covered = len({full.samples[i][1] for i in indices})
        detail = f"{covered} classes"
    else:
        detail = f"{len(full)} images in the folder"
    print(f"  calibration: {len(ds)} images from {image_dir} (stride={stride}, offset={offset}, {detail})")
    return nncf.Dataset(loader, transform_fn)


def fold_unquantized_smooth_quant(model):
    """Fold SmoothQuant scales feeding only floating-point MatMuls into weights.

    NNCF inserts an ``nncf_smooth_quant`` Multiply before every smoothed MatMul,
    including MatMuls kept in FP16 by the ignored scope. There it is a standalone
    elementwise GPU kernel. Since ``(x*s) @ W^T == x @ (W*s)^T``, the scale is
    folded into the constant weight and the Multiply bypassed. Multiplies feeding
    a quantizer (INT8 GEMMs) are left alone.
    """
    from openvino import opset13 as ops

    folded = 0
    for node in model.get_ordered_ops():
        if node.get_type_name() != "Multiply" or "nncf_smooth_quant" not in node.get_friendly_name():
            continue
        sources = [node.input(i).get_source_output() for i in range(2)]
        constants = [i for i, s in enumerate(sources) if s.get_node().get_type_name() == "Constant"]
        if len(constants) != 1:
            continue
        data = sources[1 - constants[0]]
        scale = sources[constants[0]].get_node().get_data().astype(np.float32).reshape(-1)
        targets = list(node.output(0).get_target_inputs())

        def is_float_matmul(target):
            consumer = target.get_node()
            return (
                consumer.get_type_name() == "MatMul"
                and target.get_index() == 0
                and consumer.input(1).get_source_output().get_node().get_type_name() == "Constant"
            )

        if not targets or not all(is_float_matmul(t) for t in targets):
            continue
        for target in targets:
            matmul = target.get_node()
            weight_node = matmul.input(1).get_source_output().get_node()
            weight = weight_node.get_data()
            scaled = weight.astype(np.float32) * (scale[None, :] if matmul.get_attributes()["transpose_b"] else scale[:, None])
            constant = ops.constant(scaled.astype(weight.dtype))
            constant.set_friendly_name(weight_node.get_friendly_name() + "/sq_folded")
            matmul.input(1).replace_source_output(constant.output(0))
            target.replace_source_output(data)
        folded += 1
    model.validate_nodes_and_infer_types()
    return folded


# INT8 PTQ needs a calibration set; WOQ (weights-only) never does.
calibration = None
if CALIB_DIR is not None:
    calibration = build_calibration(CALIB_DIR, CALIB_KIND, transform, CALIB_SAMPLES)
    if CALIB_KIND != "imagenet":
        print("  [int8-ptq] note: the shipped reference recipe calibrated on ImageNet-val; with COCO")
        print("             images the PTQ graph is valid but not bit-identical to that reference.")
else:
    print("  [int8-ptq] no calibration set found -> PTQ skipped (nothing is downloaded for it);")
    print("             INT8-WOQ (weights-only, no data needed) is still produced")

missing = [p for p in PRECISIONS if not ir_on_disk(ir_paths[p])]
if "int8-ptq" in missing and calibration is None:
    missing.remove("int8-ptq")
for p in PRECISIONS:
    if p in missing:
        continue
    if ir_on_disk(ir_paths[p]):
        print(f"  [{p:>8}] IR found -> skipping conversion")
    else:
        print(f"  [{p:>8}] skipped (needs calibration data, none found)")

if missing:
    if "fp32" in missing:
        print("  [fp32] tracing reference graph (FP32) ...")
        fp32_graph = ov.convert_model(patch_model, example_input=torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE))
        ir_paths["fp32"].parent.mkdir(parents=True, exist_ok=True)
        ov.save_model(fp32_graph, ir_paths["fp32"], compress_to_fp16=False)
        print("  [fp32] converted")
    if {"fp16", "int8-ptq", "int8-woq"} & set(missing):
        print("  [fp16] tracing optimized graph (FP16) ...")
        fp16_graph = ov.convert_model(opt_model, example_input=torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE, dtype=torch.float16))
    if "fp16" in missing:
        ir_paths["fp16"].parent.mkdir(parents=True, exist_ok=True)
        ov.save_model(fp16_graph, ir_paths["fp16"], compress_to_fp16=False)  # graph is already FP16
        print("  [fp16] converted")
    if "int8-ptq" in missing:
        # Shipped PTQ recipe (ov_enab/ov_quantize.py): MIXED preset, GPU target,
        # TRANSFORMER model type, fast bias correction, hybrid-b0fc1 scope — the
        # residual-feeding GEMMs (patch embed, attn proj, mlp fc2, block-0 fc1)
        # stay in FP16.
        ir_paths["int8-ptq"].parent.mkdir(parents=True, exist_ok=True)
        print(f"  [int8-ptq] quantizing (NNCF PTQ, {CALIB_SAMPLES} calibration images) ...")
        int8_ptq_graph = nncf.quantize(
            copy.deepcopy(fp16_graph),
            calibration,
            subset_size=CALIB_SAMPLES,
            preset=nncf.QuantizationPreset.MIXED,
            target_device=nncf.TargetDevice.GPU,
            model_type=nncf.ModelType.TRANSFORMER,
            fast_bias_correction=True,
            ignored_scope=nncf.IgnoredScope(patterns=[r".*patch_embed.*", r".*attn\.proj.*", r".*mlp\.fc2.*", r".*blocks\.0\.mlp\.fc1.*"]),
        )
        folds = fold_unquantized_smooth_quant(int8_ptq_graph)
        print(f"  [int8-ptq] folded {folds} SmoothQuant scales into FP16 weights")
        ov.save_model(int8_ptq_graph, ir_paths["int8-ptq"], compress_to_fp16=False)
        print("  [int8-ptq] converted")
    if "int8-woq" in missing:
        # Weights-only INT8 (no activation statistics needed): INT8 symmetric
        # weights, dynamic activation quantization in the plugin; the attention
        # output projections and block-0 fc1 stay in FP16.
        ir_paths["int8-woq"].parent.mkdir(parents=True, exist_ok=True)
        print("  [int8-woq] quantizing (NNCF weights-only INT8, no calibration data) ...")
        int8_woq_graph = nncf.compress_weights(
            copy.deepcopy(fp16_graph),
            mode=nncf.CompressWeightsMode.INT8_SYM,
            ignored_scope=nncf.IgnoredScope(patterns=[r".*attn\.proj.*", r".*blocks\.0\.mlp\.fc1.*"]),
        )
        ov.save_model(int8_woq_graph, ir_paths["int8-woq"], compress_to_fp16=False)
        print("  [int8-woq] converted")
else:
    print("\nAll IRs are cached; nothing to convert.")

# Precisions that actually have an IR on disk.
ACTIVE = [p for p in PRECISIONS if ir_on_disk(ir_paths[p])]
if len(ACTIVE) < len(PRECISIONS):
    print(f"\n  active precisions: {', '.join(ACTIVE)}")

print("\nIRs in ov_models/:")
for p in ACTIVE:
    mb = ir_paths[p].with_suffix(".bin").stat().st_size / 1e6
    print(f"  [{p:>8}] {mb:6.1f} MB   {ir_paths[p].relative_to(REPO_ROOT)}")

## 7. Run each OpenVINO model on the sample image

Compile each IR on the chosen device and run inference on the same input. For **FP32** we
force true FP32 execution (`INFERENCE_PRECISION_HINT=f32`) because the GPU plugin otherwise
runs an FP32 IR in FP16 by default — which would make the FP32 row secretly FP16.

In [ ]:
core = ov.Core()


def compile_ir(precision):
    model = core.read_model(ir_paths[precision])
    model.reshape({0: ov.PartialShape([1, 3, IMAGE_SIZE, IMAGE_SIZE])})
    config = {"INFERENCE_PRECISION_HINT": "f32"} if precision == "fp32" else {}
    return core.compile_model(model, DEVICE, config)


compiled = {}
feats_ov = {}
input_dtypes = {}
for p in ACTIVE:
    cm = compile_ir(p)
    compiled[p] = cm
    dtype = cm.input(0).get_element_type().to_dtype()  # fp16/int8 IRs take FP16 inputs
    input_dtypes[p] = dtype
    out = cm(pixel_values.numpy().astype(dtype))[cm.output(0)]  # (1, D, h, w)
    feats_ov[p] = torch.from_numpy(np.ascontiguousarray(out)).float()
    print(f"  [{p:>8}] input {np.dtype(dtype).name:>5}  output {tuple(out.shape)}")

## 8. Fidelity metrics vs the torch model

The DINO lineage characterizes its dense features with **cosine similarity**, so that is the
headline metric here — the **mean** cosine similarity over all patches — complemented by
**mean squared error** and **MAE** for the raw numerical error. Metrics are computed per patch (rows = `h*w`
patches, each a `D`-dim feature vector).

The FP32 row is a sanity check: it should be near-perfect (**cosine similarity ≈ 1.0**). FP16 and INT8 show
how much dense-feature quality each compression step costs.

In [ ]:
def feature_metrics(feat_rows, ref_rows):
    """Dense-feature fidelity of one engine vs the torch reference (rows = patches)."""
    feat, ref = feat_rows.float(), ref_rows.float()
    diff = feat - ref
    cos = F.cosine_similarity(feat, ref, dim=1)
    return {
        "cosim": cos.mean().item(),
        "mse": torch.mean(diff**2).item(),
        "mae": torch.mean(diff.abs()).item(),
    }


metrics = {p: feature_metrics(to_rows(feats_ov[p]), ref_rows) for p in ACTIVE}

hdr = f"{'prec':>8} {'cosim':>10} {'mse':>12} {'mae':>12}"
print(hdr)
print("-" * len(hdr))
for p in ACTIVE:
    m = metrics[p]
    print(f"{p:>8} {m['cosim']:>10.6f} " f"{m['mse']:>12.3e} {m['mae']:>12.3e}")

## 9. Latency

Warm up, then time `BENCH_REPEATS` single-image inferences per engine. Torch is timed on
CPU, and additionally on the Intel GPU through `torch.xpu` when the installed torch build
provides XPU support (Intel Extension for PyTorch / PyTorch XPU) — so both torch rows appear
whenever XPU is usable. The OpenVINO models run on the selected device. Reported: median
latency (ms) at batch 1.

In [ ]:
def bench_ov(cm, x_np, repeats=BENCH_REPEATS, warmup=3):
    req = cm.create_infer_request()
    for _ in range(warmup):
        req.infer(x_np)
    ts = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        req.infer(x_np)
        ts.append(time.perf_counter() - t0)
    return float(np.median(np.array(ts)) * 1e3)


def bench_torch(model, x, repeats=BENCH_REPEATS, warmup=3):
    with torch.inference_mode():
        for _ in range(warmup):
            model(x)
        ts = []
        for _ in range(repeats):
            t0 = time.perf_counter()
            model(x)
            ts.append(time.perf_counter() - t0)
        lat = float(np.median(np.array(ts)) * 1e3)
    if x.device.type == "xpu":
        # torch's caching allocator keeps the warmup/timed activations reserved on the
        # Intel GPU after measuring; while held, OpenVINO's GPU plugin runs measurably
        # slower (its FP32 graph was ~2.4x slower in the same process). Releasing the
        # cache lets the OpenVINO engines benchmark on a quiet device.
        torch.xpu.empty_cache()
    return lat


# The Intel GPU is shared with the desktop compositor, so a timing pass taken
# during unrelated graphics work can come out inflated -- in practice any engine
# can catch a slow window (FP16 and INT8 have each been observed at 2-3x their
# quiet-GPU time while the other stayed fast). Every engine is therefore measured
# in BENCH_ROUNDS interleaved rounds and the best median is kept, which recovers
# the quiet-GPU number as long as one round lands on an idle GPU -- the protocol
# behind the reference numbers in README.md.
BENCH_ROUNDS = 3

# The torch reference is benchmarked on CPU always, and additionally on the Intel
# GPU through torch.xpu when the installed torch build provides it (Intel
# Extension for PyTorch / PyTorch XPU), so both rows appear whenever XPU is usable.
import copy

torch_bench = {"cpu": (patch_model.to("cpu"), pixel_values.cpu())}
_has_xpu = hasattr(torch, "xpu") and torch.xpu.is_available()
if _has_xpu:
    torch_bench["xpu"] = (copy.deepcopy(patch_model).to("xpu"), pixel_values.to("xpu"))

perf = {}
for _round in range(BENCH_ROUNDS):
    candidates = {f"torch ({dev}, fp32)": bench_torch(m, x) for dev, (m, x) in torch_bench.items()}
    for p in ACTIVE:
        x_np = pixel_values.numpy().astype(input_dtypes[p])
        candidates[f"ov-{p} ({DEVICE})"] = bench_ov(compiled[p], x_np)
    for name, lat in candidates.items():
        perf[name] = min(perf.get(name, float("inf")), lat)

# Release the XPU copy (and any cached blocks) before the visualisation cells.
torch_bench.clear()
if _has_xpu:
    torch.xpu.empty_cache()

hdr = f"{'engine':<24} {'latency_ms':>12}"
print(hdr)
print("-" * len(hdr))
for name, lat in perf.items():
    print(f"{name:<24} {lat:>12.2f}")

## 10. Visualization — PCA & segmentation (torch vs OpenVINO)

Four rows, one column per engine (torch, FP32, FP16, INT8):

1. **input** — the pixels the model sees.
2. **PCA-RGB** — top-3 PCA of the dense features (basis fit on the torch reference and reused
   everywhere, so colors are directly comparable).
3. **segmentation** — k-means over the patch features (clusters fit on torch, predicted for
   each precision) → semantic regions.
4. **cosine similarity** — per-patch cosine similarity to the torch features (the measure
   DINOv2 and its lineage use to compare dense features, matching the metrics table above).
   A shared color scale makes the panels comparable; brighter = closer to torch, so any
   precision drop shows up as darker patches.

In [ ]:
from sklearn.cluster import KMeans

# --- PCA basis: fit ONCE on the torch reference and reuse for every engine ----
# torch.pca_lowrank is randomized, so recomputing it per column would flip the
# sign/rotation and change colours. Fixing one basis makes the columns directly
# comparable (the same feature direction -> the same colour everywhere), so the
# FP32 panel matches torch exactly.
_mean = ref_rows.mean(0, keepdim=True)
_, _, _v = torch.pca_lowrank(ref_rows - _mean, q=3)
_basis = _v[:, :3]
_proj_ref = (ref_rows - _mean) @ _basis
_lo, _hi = _proj_ref.min(0).values, _proj_ref.max(0).values


def pca_rgb(feat_bdhw):
    proj = (to_rows(feat_bdhw) - _mean) @ _basis
    rgb = ((proj - _lo) / (_hi - _lo + 1e-8)).clamp(0, 1)
    return rgb.reshape(GRID, GRID, 3).cpu().numpy()


# --- segmentation: KMeans fit on torch, predicted for every precision ---------
N_SEG = 5
km = KMeans(n_clusters=N_SEG, n_init=10, random_state=0).fit(ref_rows.numpy())


def seg_grid(feat_bdhw):
    return km.predict(to_rows(feat_bdhw).numpy()).reshape(GRID, GRID)


# --- per-patch cosine similarity to the torch reference (DINO's dense measure)
def cos_sim_grid(feat_bdhw):
    hwd = feat_bdhw[0].movedim(0, -1).float()
    return F.cosine_similarity(hwd, ref_hwd, dim=-1).numpy()  # (h, w)


def upsample(grid, size, mode="nearest"):
    t = torch.from_numpy(grid.astype(np.float32))[None, None]
    return F.interpolate(t, size=(size, size), mode=mode)[0, 0].numpy()


engines = ["torch"] + list(ACTIVE)
feat_all = {"torch": ref_torch, **feats_ov}
row_titles = ["input", "PCA-RGB", f"segmentation (k={N_SEG})", "cosine sim vs torch"]

# Cosine-similarity maps; the colour range starts at the worst patch across the
# OpenVINO engines (torch's self-map is all 1.0) so the panels stay comparable
# and any precision loss is visible rather than washed out.
sim_grids = {eng: cos_sim_grid(feat_all[eng]) for eng in engines}
SIM_VMIN = min(float(sim_grids[p].min()) for p in ACTIVE)

fig, axes = plt.subplots(4, len(engines), figsize=(3.3 * len(engines), 12.2))
for j, eng in enumerate(engines):
    axes[0, j].imshow(display_np)
    axes[0, j].set_title(eng)
    axes[0, j].axis("off")
    axes[1, j].imshow(pca_rgb(feat_all[eng]))
    axes[1, j].axis("off")
    seg = upsample(seg_grid(feat_all[eng]), IMAGE_SIZE)
    axes[2, j].imshow(display_np)
    axes[2, j].imshow(seg, cmap="tab10", alpha=0.55)
    axes[2, j].axis("off")
    sim = sim_grids[eng]
    axes[3, j].imshow(display_np)
    hm = axes[3, j].imshow(upsample(sim, IMAGE_SIZE), cmap="viridis", alpha=0.7, vmin=SIM_VMIN, vmax=1.0)
    axes[3, j].set_title(f"cosim={sim.mean():.4f}")
    axes[3, j].axis("off")

for i, title in enumerate(row_titles):
    axes[i, 0].set_ylabel(title, rotation=90, labelpad=14, fontsize=11)
    axes[i, 0].axis("on")
    axes[i, 0].set_xticks([])
    axes[i, 0].set_yticks([])
    for spine in axes[i, 0].spines.values():
        spine.set_visible(False)

fig.suptitle(f"DINOv2-S/14-Reg dense features — torch vs OpenVINO ({DEVICE}) @ {IMAGE_SIZE}px", y=1.005)
fig.tight_layout()
fig.colorbar(hm, ax=axes[3, :].tolist(), fraction=0.015, pad=0.01, label="cosine similarity to torch")
plt.show()

## 11. Summary

A compact recap of fidelity, latency and IR sizes, plus where the cached artifacts
live — the next run reuses all of them and jumps straight to inference.

In [ ]:
print("=" * 72)
print(f"DINOv2-S/14-Reg - OpenVINO FP32 / FP16 / INT8-PTQ / INT8-WOQ  (device={DEVICE}, {IMAGE_SIZE}px)")
print("=" * 72)

print("\nFidelity vs torch (dense patch features):")
for p in ACTIVE:
    m = metrics[p]
    print(f"  {p:>8}: cosim={m['cosim']:.5f}  " f"mse={m['mse']:.3e}  mae={m['mae']:.3e}")

print("\nLatency:")
for name, lat in perf.items():
    print(f"  {name:<24} {lat:7.2f} ms")

print("\nIR sizes:")
for p in ACTIVE:
    mb = ir_paths[p].with_suffix(".bin").stat().st_size / 1e6
    print(f"  {p:>8}: {mb:.1f} MB")

print("\nCached artifacts (re-runs skip download + conversion):")
print(f"  weights : {CHECKPOINT_DIR.relative_to(REPO_ROOT)}")
print(f"  image   : {SAMPLE_PATH.relative_to(REPO_ROOT)}")
print(f"  IRs     : {OV_DIR.relative_to(REPO_ROOT)}")